<a href="https://www.kaggle.com/code/pcsciprav/chess-bot-fun?scriptVersionId=369155887" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Working Crackfish

## 1. Get packages, Lc0, and base Stockfish

In [ ]:
%%bash
sudo apt-get update
sudo apt-get -y install git python3-pip gcc-16 g++-16 zlib1g zlib1g-dev ninja-build
pip3 install meson --break-system-packages

In [ ]:
%%bash
cd /kaggle/working
#rm -rf stockfish* lc0* match_results.pgn
# 3. Download the actual, fully-compiled STOCKFISH 19 Linux binary
wget https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz
tar -xf stockfish-linux-x86-64-universal.tar.gz
chmod +x /kaggle/working/stockfish/stockfish-linux-x86-64-universal

In [ ]:
%%bash
git clone --depth 1 https://github.com/LeelaChessZero/lc0.git /kaggle/working/lc0
cd /kaggle/working/lc0
git submodule update --init --recursive
wget https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
export PATH=/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH

In [ ]:
!/kaggle/working/lc0/build.sh release

In [ ]:
%%bash
git clone --depth 1 https://github.com/lucasart/c-chess-cli.git /kaggle/working/c-chess-cli
cd /kaggle/working/c-chess-cli
python3 make.py

## 2. Run Lc0 for training data

In [ ]:
!chmod +x /kaggle/working/lc0/build/release/lc0

In [5]:
%%bash

cat > /kaggle/working/lc0_gpu0.sh <<'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=0
exec /kaggle/working/lc0/build/release/lc0 "$@"
EOF

cat > /kaggle/working/lc0_gpu1.sh <<'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=1
exec /kaggle/working/lc0/build/release/lc0 "$@"
EOF
cat << 'EOF' > process1.sh 

chmod +x /kaggle/working/lc0_gpu0.sh /kaggle/working/lc0_gpu1.sh

RUY_LOPEZ_FEN="rnbqkbnr/pppppppp/8/8/8/N7/PPPPPPPP/R1BQKBNR b KQkq - 1 1"
echo "${RUY_LOPEZ_FEN}" > /kaggle/working/ruy_lopez_start.epd

/kaggle/working/c-chess-cli/c-chess-cli \
  -each tc=50/1 \
  -engine name=Lc0_toolow cmd=/kaggle/working/lc0_gpu0.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=0.01 \
    option.PolicyTemperature=0.00005 \
    option.SmartPruningFactor=0.25 \
    option.CPuctFactor=0.1 \
  -engine name=Lc0_superlow cmd=/kaggle/working/lc0_gpu0.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=0 \
    option.PolicyTemperature=0.04 \
    option.SmartPruningFactor=5.0 \
    option.CPuctFactor=0.25 \
  -openings file=/kaggle/working/ruy_lopez_start.epd \
  -repeat \
  -rounds 500 \
  -pgn /kaggle/working/lc0basetrain.pgn
EOF
cat << 'EOF' > process2.sh 
chmod +x /kaggle/working/lc0_gpu0.sh /kaggle/working/lc0_gpu1.sh

RUY_LOPEZ_FEN="rnbqk2r/pppp1ppp/5n1B/2b1p3/3P4/3Q4/PPP1PPPP/RN2KBNR w KQkq - 2 4"
echo "${RUY_LOPEZ_FEN}" > /kaggle/working/ruy_lopez_start.epd

/kaggle/working/c-chess-cli/c-chess-cli \
  -each tc=50/1 \
  -engine name=Lc0_maxtemp cmd=/kaggle/working/lc0_gpu1.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=1904.4 \
    option.PolicyTemperature=240.00214 \
    option.SmartPruningFactor=5.0 \
    option.CPuctFactor=0.04 \
    option.FpuStrategy=reduction \
    option.FpuValue=0.05 \
-engine name=Lc0_hightemp cmd=/kaggle/working/lc0_gpu1.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=20 \
    option.PolicyTemperature=4 \
    option.SmartPruningFactor=1.0 \
    option.CPuctFactor=1 \
  -openings file=/kaggle/working/ruy_lopez_start.epd \
  -repeat \
  -rounds 500 \
  -pgn /kaggle/working/lc0basetrain.pgn
EOF

In [2]:
!chmod +x /kaggle/working/lc0/build/release/lc0 

In [16]:
!printf 'uci\nquit\n' | /kaggle/working/lc0/build/release/lc0  | grep -i fpu

       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classic
option name FpuStrategy type combo default reduction var reduction var absolute
option name FpuValue type string default 0.330000


In [7]:
!bash process1.sh & bash process2.sh & wait

       _       _

|   _ | ||   _ | |

|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classicSearch algorithm: classic

       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classic
Search algorithm: classic
[1] Started game 1 of 500 (Lc0_superlow vs Lc0_toolow)
[1] Started game 1 of 500 (Lc0_hightemp vs Lc0_maxtemp)
Loading weights file from: /kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
Loading weights file from: /kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
Weights file has multihead format, updating format flag
Weights file has multihead format, updating format flag
CUDA Runtime version: 12.8.0
CUDA Runtime version: 12.8.0
Latest version of CUDA supported by the driver: 13.0.0
GPU: Tesla T4
GPU memory: 14.5622 Gb
GPU clock frequency: 1590 MHz
GPU compute capab

In [85]:
%%bash
chmod +x /kaggle/working/lc0_gpu0.sh /kaggle/working/lc0_gpu1.sh
chmod +x /kaggle/working/crackfish1924

RUY_LOPEZ_FEN="rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"
echo "${RUY_LOPEZ_FEN}" > /kaggle/working/start.epd

/kaggle/working/c-chess-cli/c-chess-cli \
 -each movetime=5 \
 -engine name=Lc0_base cmd=/kaggle/working/lc0/build/release/lc0 \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
 -engine name=Crackfish1924 cmd=/kaggle/working/crackfish1924 \
    option.Hash=4096 \
    option.Threads=4 \
    option.EvalFile=/kaggle/working/crackedfish.nnue \
  -openings file=/kaggle/working/start.epd \
  -repeat \
  -rounds 2 \
  -pgn /kaggle/working/lac0vcrackfish.pgn \

[1] Started game 1 of 2 (Lc0_base vs Crackfish1924)
[1] Finished game 1 (Lc0_base vs Crackfish1924): 1-0 {time forfeit}
Score of Lc0_base vs Crackfish1924: 1 - 0 - 0  [1.000] 1
[1] Started game 2 of 2 (Lc0_base vs Crackfish1924)
[1] Finished game 2 (Lc0_base vs Crackfish1924): 1-0 {time forfeit}
Score of Lc0_base vs Crackfish1924: 2 - 0 - 0  [1.000] 2


       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classic
Loading weights file from: /kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
Weights file has multihead format, updating format flag
CUDA Runtime version: 12.8.0
Latest version of CUDA supported by the driver: 13.0.0
GPU: Tesla T4
GPU memory: 14.5622 Gb
GPU clock frequency: 1590 MHz
GPU compute capability: 7.5
L2 cache capacity: 4194304


In [ ]:
!tail -n 100 /kaggle/working/lc0basetrain.pgn

In [86]:
!cat /kaggle/working/lac0vcrackfish.pgn

[Round "1.1"]
[White "Lc0_hightemp"]
[Black "Crackfish1924"]
[Result "0-1"]
[Termination "time forfeit"]
[FEN "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"]
[PlyCount "0"]

0-1

[Round "2.1"]
[White "Lc0_hightemp"]
[Black "Crackfish1924"]
[Result "0-1"]
[Termination "time forfeit"]
[FEN "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"]
[PlyCount "0"]

0-1

[Round "1.1"]
[White "Lc0_hightemp"]
[Black "Crackfish1924"]
[Result "0-1"]
[Termination "time forfeit"]
[FEN "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"]
[PlyCount "0"]

0-1

[Round "2.1"]
[White "Lc0_hightemp"]
[Black "Crackfish1924"]
[Result "0-1"]
[Termination "time forfeit"]
[FEN "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"]
[PlyCount "0"]

0-1

[Round "1.1"]
[White "Lc0_hightemp"]
[Black "Crackfish1924"]
[Result "0-1"]
[Termination "time forfeit"]
[FEN "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"]
[PlyCount "0"]

0-1

[Round "2.1"]
[White "Lc0

## 3. Make a binpack for running a NNUE train

In [8]:
!pip -q install python-chess tyro

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 84.3 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 13.1 MB/s eta 0:00:00


In [9]:
import chess
import chess.pgn
import random
from collections import Counter

PGN = "/kaggle/working/lc0basetrain.pgn"
TRAIN_TXT = "/kaggle/working/cf_train.txt"
VAL_TXT = "/kaggle/working/cf_val.txt"

PLY_STRIDE = 1
MIN_GAME_PLY = 4
SKIP_LAST_PLIES = 2
VAL_FRACTION = 0.10
MAX_PER_POSITION = 5
SEED = 196342

rng = random.Random(SEED)

# --------------------------------------------------
# Load completed games
# --------------------------------------------------

games = []
skipped = 0

with open(PGN, encoding="utf-8-sig") as f:
    while True:
        game = chess.pgn.read_game(f)

        if game is None:
            break

        if game.headers.get("Result") not in (
            "1-0", "0-1", "1/2-1/2"
        ):
            skipped += 1
            continue

        games.append(game)

print(f"Loaded {len(games)} completed games")
print(f"Skipped {skipped} unfinished games")

# --------------------------------------------------
# Split by GAME, not by position
# --------------------------------------------------

indices = list(range(len(games)))
rng.shuffle(indices)

n_val_games = max(
    1, round(len(indices) * VAL_FRACTION)
)

val_games = set(indices[:n_val_games])

# --------------------------------------------------
# Extract positions
# --------------------------------------------------

def result_value(game):
    result = game.headers["Result"]

    if result == "1-0":
        return 1
    if result == "0-1":
        return -1
    return 0


def extract(game, game_id):
    board = game.board()
    moves = list(game.mainline_moves())
    result = result_value(game)

    records = []

    for ply, move in enumerate(moves):

        if (
            MIN_GAME_PLY <= ply
            and ply < len(moves) - SKIP_LAST_PLIES
            and ply % PLY_STRIDE == 0
        ):
            # Result from the side-to-move perspective
            pov_result = (
                result
                if board.turn == chess.WHITE
                else -result
            )

            records.append((
                board.fen(),
                move.uci(),
                ply,
                game_id,
                pov_result
            ))

        board.push(move)

    return records


train_raw = []
val_raw = []

for i, game in enumerate(games):
    records = extract(game, i)

    if i in val_games:
        val_raw.extend(records)
    else:
        train_raw.extend(records)

print(
    f"Raw positions: "
    f"{len(train_raw)} train / "
    f"{len(val_raw)} val"
)

# --------------------------------------------------
# Position identity
# Ignore halfmove/fullmove counters
# --------------------------------------------------

def position_key(fen):
    return " ".join(fen.split()[:4])

# --------------------------------------------------
# Keep up to N examples per position
# --------------------------------------------------

def dedupe(records, forbidden=None, max_per_position=5):
    if forbidden is None:
        forbidden = set()

    records = records.copy()
    rng.shuffle(records)

    counts = Counter()
    seen = set()
    output = []

    for record in records:
        fen = record[0]
        key = position_key(fen)

        # Prevent exact-position overlap
        # between training and validation
        if key in forbidden:
            continue

        if counts[key] >= max_per_position:
            continue

        counts[key] += 1
        seen.add(key)
        output.append(record)

    return output, seen


train, train_keys = dedupe(
    train_raw,
    max_per_position=MAX_PER_POSITION
)

val, val_keys = dedupe(
    val_raw,
    forbidden=train_keys,
    max_per_position=MAX_PER_POSITION
)

# --------------------------------------------------
# Write txt2binpack-compatible records
# --------------------------------------------------

def write_records(records, path):
    written = 0
    invalid = 0

    with open(path, "w") as f:
        for fen, move, ply, game_id, result in records:
            board = chess.Board(fen)
            move_obj = chess.Move.from_uci(move)

            if move_obj not in board.legal_moves:
                invalid += 1
                continue

            # Same labeling convention as your old code
            cp = result * 1000

            f.write(
                f"{fen}|{move}|{cp}|{ply}|{result}\n"
            )

            written += 1

    return written, invalid


n_train, bad_train = write_records(
    train, TRAIN_TXT
)

n_val, bad_val = write_records(
    val, VAL_TXT
)

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n========== DATASET SUMMARY ==========")
print(f"Games: {len(games)}")
print(f"Training games: {len(games) - len(val_games)}")
print(f"Validation games: {len(val_games)}")

print(f"\nTrain records: {n_train:,}")
print(f"Validation records: {n_val:,}")

print(f"\nUnique train positions: {len(train_keys):,}")
print(f"Unique validation positions: {len(val_keys):,}")

print(f"\nInvalid train moves: {bad_train}")
print(f"Invalid validation moves: {bad_val}")

print(f"\nPosition overlap: {len(train_keys & val_keys)}")
print(f"Max examples per position: {MAX_PER_POSITION}")

print(f"\nTrain: {TRAIN_TXT}")
print(f"Validation: {VAL_TXT}")
print("=====================================")


Loaded 9647 completed games
Skipped 0 unfinished games
Raw positions: 716863 train / 80244 val

========== DATASET SUMMARY ==========
Games: 9647
Training games: 8682
Validation games: 965

Train records: 105,164
Validation records: 2,280

Unique train positions: 45,740
Unique validation positions: 2,127

Invalid train moves: 0
Invalid validation moves: 0

Position overlap: 0
Max examples per position: 5

Train: /kaggle/working/cf_train.txt
Validation: /kaggle/working/cf_val.txt


In [10]:
!/kaggle/working/nnue-pytorch/txt2binpack \
  /kaggle/working/cf_train.txt \
  /kaggle/working/cf_train.binpack

!/kaggle/working/nnue-pytorch/txt2binpack \
  /kaggle/working/cf_val.txt \
  /kaggle/working/cf_val.binpack

!ls -lh /kaggle/working/cf_train.binpack /kaggle/working/cf_val.binpack

Written:  105164
Rejected: 0
Written:  2280
Rejected: 0
-rw-r--r-- 1 root root 3.5M Nov 13 12:20 /kaggle/working/cf_train.binpack
-rw-r--r-- 1 root root  76K Nov 13 12:20 /kaggle/working/cf_val.binpack


In [ ]:
%cd /kaggle/working/nnue-pytorch

!bash compile_data_loader.bat

In [ ]:
%cd /kaggle/working/nnue-pytorch

!find build -name "libtraining_data_loader.so" -print
!python train.py --help

## 4. Begin Training

In [11]:
%cd /kaggle/working/nnue-pytorch
#!pip install tyro
!python train.py \
    /kaggle/working/cf_train.binpack \
    --validation-datasets /kaggle/working/cf_val.binpack \
    --validation-size 2280 \
    --check-val-every-n-epoch 1 \
    --accelerator cuda \
    --gpus 0 \
    --batch-size 16384 \
    --epoch-size 105164 \
    --max-epochs 6 \
    --lr 0.0003 \
    --network-save-period 160 \
    --save-last-network True \
    --default-root-dir /kaggle/working/crackedfish_nnue \
    --lambda 0.0 \
    --warmup-steps 10 \
    --no-filtered \
    --no-wld-filtered \
    --soft-early-fen-skipping 0


/kaggle/working/nnue-pytorch
batch_size(global)=16384 | n_devices=1 | batch_size(per_gpu)=16384
Loss parameters:
LossParams(in_offset=270, out_offset=270, in_scaling=340, out_scaling=380, lambda_config=LambdaConfig(start_lambda=1.0, end_lambda=1.0, jitter_lambda_sample=0.0, jitter_lambda_batch=0.0, jitter_decay_lambda_batch=0.0, lambda_=0.0, lambda_schedule_steps=-1, lambda_cycle_warmup_pct=0.3, lambda_cycle_delta=0.0, lambda_cycle_jitter=False), pow_exp=2.5, qp_asymmetry=0.0, w1=0.0, w2=0.5, tb_remap_base=4000.0, tb_remap_scale=4000.0, tb_remap_decay=0.85)
Feature set: Full_Threats+PP_3Wide+HalfKAv2_hm^
Num inputs: 88944
Training with: ('/kaggle/working/cf_train.binpack',)
Validating with: ('/kaggle/working/cf_val.binpack',)
Seed 42
DataloaderSkipConfig(filtered=False, wld_filtered=False, random_fen_skipping=0, early_fen_skipping=-1, soft_early_fen_skipping=0, simple_eval_skipping=-1, param_index=0, pc_y0=0.0, pc_y1=0.4, pc_y2=1.0, pc_y3=1.0, pc_y4=0.75, ply_x1=0.0, ply_y1=0.1, ply_x2

In [ ]:
!find /kaggle/working/crackedfish_nnue -type f \
  -printf '%s %p\n' | sort -

In [12]:
!ls /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints

last.ckpt


In [13]:
%%bash
# Create a safe backup of your true last checkpoint in the root directory
cp /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints/last.ckpt /kaggle/working/Crackfish1924_final.ckpt

# Purge the cluttered logging workspace entirely
rm -rf /kaggle/working/crackedfish_nnue

# Verify that only your clean master checkpoint remains
ls -lh /kaggle/working/


total 2.1G
drwxr-xr-x  6 root root 4.0K Nov 13 11:58 c-chess-cli
-rw-r--r--  1 root root 3.5M Nov 13 12:20 cf_train.binpack
-rw-r--r--  1 root root 6.3M Nov 13 12:20 cf_train.txt
-rw-r--r--  1 root root  76K Nov 13 12:20 cf_val.binpack
-rw-r--r--  1 root root 130K Nov 13 12:20 cf_val.txt
-rw-r--r--  1 root root  87M Nov 13 11:57 crackedfish.nnue
-rw-r--r--  1 root root  93K Nov 13 11:58 crackedfish_search_backup.cpp
-rw-r--r--  1 root root  95M Nov 13 11:57 crackfish1924
-rw-r--r--  1 root root 1.8G Nov 13 12:40 Crackfish1924_final.ckpt
-rw-r--r--  1 root root 6.1K Nov 13 11:58 isitbetteronmoreram.pgn
-rw-r--r--  1 root root 2.4K Nov 13 11:58 lac0vcrackfish.pgn
-rw-r--r--  1 root root  11K Nov 13 11:58 lac0vscrackfish.pgn
drwxr-xr-x 14 root root 4.0K Nov 13 11:58 lc0
-rw-r--r--  1 root root  18M Nov 13 12:19 lc0basetrain.pgn
-rwxr-xr-x  1 root root   90 Nov 13 11:59 lc0_gpu0.sh
-rwxr-xr-x  1 root root   90 Nov 13 11:59 lc0_gpu1.sh
-rw-r--r--  1 root root  18K Nov 13 11:58 madness.pgn
-

In [40]:
import torch

p = "/kaggle/working/Crackfish1924_final.ckpt"

ckpt = torch.load(p, map_location="cpu", weights_only=False)

print("Checkpoint loaded successfully")
print("epoch:", ckpt.get("epoch"))
print("global_step:", ckpt.get("global_step"))
print("keys:", ckpt.keys())

Checkpoint loaded successfully
epoch: 5
global_step: 36
keys: dict_keys(['epoch', 'global_step', 'state_dict', 'lr_schedulers', 'jitter_buffer_value'])


In [41]:
%cd /kaggle/working/nnue-pytorch

!python serialize.py \
  "/kaggle/working/Crackfish1924_final.ckpt" \
  "/kaggle/working/crackedfish.nnue" \
  --description "Crackedfish cursed NNUE"
!ls -lh /kaggle/working/crackedfish.nnue
!sha256sum /kaggle/working/crackedfish.nnue
!chmod +x /kaggle/working/crackfish1924
!printf 'uci\nsetoption name EvalFile value /kaggle/working/crackedfish.nnue\nisready\nposition startpos\ngo depth 1\nquit\n' | /kaggle/working/crackfish1924 2>&1

/kaggle/working/nnue-pytorch
Converting /kaggle/working/Crackfish1924_final.ckpt to /kaggle/working/crackedfish.nnue
---------------
ft_bias: 
-5        | 
-3        | 
-1        | ###
0         | #########################################
1         | ##################################################
2         | ###
4         | 
Number of elements: 1024
Minimum value in layer is -1, occurring 35 times.
Maximum value in layer is 2, occurring 34 times.
---------------
---------------
Full_Threats ft_weight: 
-6        | 
-4        | 
-2        | #################
0         | ##################################################
2         | 
4         | 
Number of elements: 61243392
Minimum value in layer is -2, occurring 997 times.
Maximum value in layer is 2, occurring 6378 times.
---------------
---------------
Full_Threats psqt_weight: 
-25       | 
-17       | 
-10       | 
-3        | ##################################################
3         | 
10        | 
17        | 
Number of el

In [ ]:
%%bash
printf 'uci\nquit\n' | /kaggle/working/stockfish/stockfish-linux-x86-64-universal

# Lc0: print every UCI option
printf 'uci\nquit\n' | /kaggle/working/lc0/build/release/lc0

## 5. Madness code

In [ ]:
!sed -i 's/optimism\[us\]  = 114 \* avg/optimism[us]  = 1924 * avg/' \
    /kaggle/working/stockfish/src/search.cpp

In [ ]:
!sed -i \
's/Depth R = 7 + depth \/ 3 + std::max((ss->staticEval - beta) \/ 256, 0);/Depth R = 1 + depth \/ 64 + std::max((ss->staticEval - beta) \/ 919191, 0);/' \
/kaggle/working/stockfish/src/search.cpp

!grep -n -A2 -B2 "Depth R =" /kaggle/working/stockfish/src/search.cpp

In [28]:
%%bash
cd /kaggle/working/stockfish/src
make clean
cp /kaggle/working/crackedfish.nnue /kaggle/working/stockfish/src/nn.nnue
make build ARCH=x86-64 EvalFile=/kaggle/working/stockfish/src/nn.nnue -j$(nproc)
chmod +x /kaggle/working/crackfish1924

Existing nn-1a298aa575a0.nnue validated, skipping download

Config:
debug: 'no'
sanitize: 'none'
optimize: 'yes'
arch: 'x86_64'
bits: '64'
kernel: 'Linux'
os: 'GNU/Linux'
prefetch: 'yes'
popcnt: 'no'
pext: 'no'
sse: 'yes'
mmx: 'no'
sse2: 'yes'
ssse3: 'no'
sse41: 'no'
avx2: 'no'
avxvnni: 'no'
avx512: 'no'
vnni512: 'no'
avx512icl: 'no'
altivec: 'no'
vsx: 'no'
neon: 'no'
dotprod: 'no'
arm_version: '0'
lsx: 'no'
lasx: 'no'
syzygy: 'yes'
target_windows: ''

Flags:
CXX: g++
CXXFLAGS:  -Wall -Wcast-qual -fno-exceptions -std=c++17  -pedantic -Wextra -Wshadow -Wmissing-declarations -m64 -Wstack-usage=128000 -fno-ipa-cp-clone -fconstexpr-ops-limit=500000000 -DNDEBUG -O3 -funroll-loops -DIS_64BIT -msse -DUSE_SSE2 -msse2 -DARCH=x86-64 -flto -flto-partition=one
LDFLAGS:   -m64 -Wl,--no-as-needed -lpthread -lrt  -Wall -Wcast-qual -fno-exceptions -std=c++17  -pedantic -Wextra -Wshadow -Wmissing-declarations -m64 -Wstack-usage=128000 -fno-ipa-cp-clone -fconstexpr-ops-limit=500000000 -DNDEBUG -O3 -funr

In file included from attacks.cpp:19:
attacks.h:27:10: fatal error: types.h: No such file or directory
   27 | #include "types.h"
      |          ^~~~~~~~~
compilation terminated.
make[1]: *** [Makefile:1318: attacks.o] Error 1
make[1]: *** Waiting for unfinished jobs....
In file included from bitboard.cpp:19:
bitboard.h:28:10: fatal error: types.h: No such file or directory
   28 | #include "types.h"
      |          ^~~~~~~~~
compilation terminated.
In file included from evaluate.cpp:19:
evaluate.h:24:10: fatal error: types.h: No such file or directory
   24 | #include "types.h"
      |          ^~~~~~~~~
compilation terminated.
make[1]: *** [Makefile:1318: bitboard.o] Error 1
make[1]: *** [Makefile:1318: evaluate.o] Error 1
In file included from history.h:32,
                 from engine.h:34,
                 from benchmark.cpp:20:
memory.h:33:10: fatal error: types.h: No such file or directory
   33 | #include "types.h"
      |          ^~~~~~~~~
compilation terminated.
make[1]: 

In [27]:
!sudo apt-get install types.h

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'node-types-handlebars' for regex 'types.h'
Note, selecting 'libghc-persistable-types-hdbc-pg-prof-0.0.3.5-222c0' for regex 'types.h'
Note, selecting 'python3-types-html5lib' for regex 'types.h'
Note, selecting 'libghc-persistable-types-hdbc-pg-prof' for regex 'types.h'
Note, selecting 'node-types-h264-profile-level-id' for regex 'types.h'
Note, selecting 'libghc-persistable-types-hdbc-pg-dev-0.0.3.5-222c0' for regex 'types.h'
Note, selecting 'libghc-persistable-types-hdbc-pg-dev' for regex 'types.h'
Note, selecting 'libghc-persistable-types-hdbc-pg-doc' for regex 'types.h'
Note, selecting 'node-types-http-errors' for regex 'types.h'
Note, selecting 'node-types-hooker' for regex 'types.h'
Note, selecting 'node-types-has-ansi' for regex 'types.h'
Note, selecting 'python3-types-httplib2' for regex 'types.h'
Note, selecting 'node-types-http-cache-semantics' for regex 'types.h'


In [57]:
%%bash
set -e

cd /kaggle/working

# Preserve your modified search.cpp
cp stockfish/src/search.cpp crackedfish_search_backup.cpp

# Clone a complete Stockfish 19 source tree
git clone --depth 1 --branch sf_19 \
    https://github.com/official-stockfish/Stockfish.git stockfish_fresh

# Reapply Crackedfish's modifications to the fresh source
python3 - <<'PY'
from pathlib import Path

p = Path("/kaggle/working/stockfish_fresh/src/search.cpp")
s = p.read_text()

changes = [
    ("114 * avg", "1924 * avg"),
    ("7 + depth / 3 + std::max((ss->staticEval - beta) / 256, 0)",
     "1 + depth / 64 + std::max((ss->staticEval - beta) / 919191, 0)"),
    ("2872 / 128.0 * std::log(i)", "14 / 1280.0 * std::log(i)"),
]

for old, new in changes:
    if old not in s:
        raise RuntimeError(f"Could not find expected code: {old}")
    s = s.replace(old, new, 1)

p.write_text(s)
PY

cd stockfish_fresh/src

# Build with a broadly compatible CPU target
make -j"$(nproc)" build ARCH=x86-64 EXE=crackfish1924

# Check the resulting binary
printf 'uci\nquit\n' | ./crackfish1924

Stockfish 19 by the Stockfish developers (see AUTHORS file)
id name Stockfish 19
id author the Stockfish developers (see AUTHORS file)

option name Debug Log File type string default <empty>
option name NumaPolicy type string default auto
option name Threads type spin default 1 min 1 max 1024
option name Hash type spin default 16 min 1 max 33554432
option name Clear Hash type button
option name Ponder type check default false
option name MultiPV type spin default 1 min 1 max 256
option name Skill Level type spin default 20 min 0 max 20
option name Move Overhead type spin default 10 min 0 max 5000
option name nodestime type spin default 0 min 0 max 10000
option name UCI_Chess960 type check default false
option name UCI_LimitStrength type check default false
option name UCI_Elo type spin default 1320 min 1320 max 3190
option name UCI_ShowWDL type check default false
option name SyzygyPath type string default <empty>
option name SyzygyProbeDepth type spin default 1 min 1 max 100
option na

In [58]:
import subprocess

engine = subprocess.Popen(
    ["/kaggle/working/stockfish_fresh/src/crackfish1924"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def send(command):
    engine.stdin.write(command + "\n")
    engine.stdin.flush()

send("setoption name EvalFile value /kaggle/working/crackedfish.nnue")
send("isready")

for line in engine.stdout:
    print(line.rstrip())
    if line.strip() == "readyok":
        break

send("position startpos")
send("go depth 4")

for line in engine.stdout:
    print(line.rstrip())
    if line.startswith("bestmove"):
        break

send("quit")
engine.wait(timeout=10)

Stockfish 19 by the Stockfish developers (see AUTHORS file)
readyok
info string Available processors: 0-3
info string Using 1 thread
info string NNUE evaluation using /kaggle/working/crackedfish.nnue (109MiB, (86896, 1024, 32, 32, 1))
info string Network replica 1: Shared memory.
info depth 1 seldepth 2 multipv 1 score cp -43 nodes 20 nps 20000 hashfull 0 tbhits 0 time 1 pv c2c3
info depth 2 seldepth 3 multipv 1 score cp -18 nodes 47 nps 47000 hashfull 0 tbhits 0 time 1 pv c2c3
info depth 3 seldepth 4 multipv 1 score cp -21 nodes 257 nps 128500 hashfull 0 tbhits 0 time 2 pv g2g3 c7c6
info depth 4 seldepth 5 multipv 1 score cp -14 nodes 304 nps 152000 hashfull 0 tbhits 0 time 2 pv g1f3 c7c6
bestmove g1f3 ponder c7c6


0

In [ ]:
!sed -i \
's/reductions\[i\] = int(2872 \/ 128.0 \* std::log(i));/reductions[i] = int(14 \/ 1280.0 * std::log(i));/' \
/kaggle/working/stockfish/src/search.cpp

!grep -n "reductions\[i\]" /kaggle/working/stockfish/src/search.cpp

In [ ]:
!cd /kaggle/working && git clone https://github.com/official-stockfish/nnue-pytorch.git
!cd nnue-pytorch

In [ ]:
import subprocess

SF = "/kaggle/working/stockfish/stockfish-linux-x86-64-universal"

p = subprocess.run(
    [SF],
    input="uci\nquit\n",
    text=True,
    capture_output=True
)

print(p.stdout)

In [ ]:
%%bash
cd /kaggle/working/nnue-pytorch
#find . -maxdepth 3 -type f | sort | head -150
ls -lah data_loader
find data_loader -maxdepth 3 -type f | sort

In [ ]:
!grep -nE "fromFen|fromFEN|setFen|setFEN|fen|FEN|fromUci|fromUCI|uci|UCI" \
/kaggle/working/nnue-pytorch/data_loader/cpp/lib/chess.h | head -100

In [ ]:
from pathlib import Path

code = r'''
#include <iostream>
#include <fstream>
#include <sstream>
#include <string>
#include <cstdint>

#include "data_loader/cpp/lib/binpack.h"
#include "data_loader/cpp/lib/chess.h"
#include "data_loader/cpp/lib/training_data_entry.h"

static std::string trim(std::string s)
{
    const auto first = s.find_first_not_of(" \t\r\n");
    if (first == std::string::npos)
        return "";

    const auto last = s.find_last_not_of(" \t\r\n");
    return s.substr(first, last - first + 1);
}

int main(int argc, char** argv)
{
    if (argc != 3)
    {
        std::cerr
            << "Usage: txt2binpack INPUT.txt OUTPUT.binpack\n";
        return 1;
    }

    std::ifstream input(argv[1]);

    if (!input)
    {
        std::cerr << "Could not open input file.\n";
        return 1;
    }

    // Truncate so rerunning the Kaggle cell doesn't append
    // another copy to an existing dataset.
    binpack::CompressedTrainingDataEntryWriter writer(
        argv[2],
        std::ios_base::trunc
    );

    std::string line;
    std::size_t lineNumber = 0;
    std::size_t written = 0;
    std::size_t rejected = 0;

    while (std::getline(input, line))
    {
        ++lineNumber;

        if (line.empty())
            continue;

        // Format:
        // FEN|MOVE|SCORE|PLY|RESULT

        std::stringstream ss(line);

        std::string fen;
        std::string moveText;
        std::string scoreText;
        std::string plyText;
        std::string resultText;

        if (!std::getline(ss, fen, '|') ||
            !std::getline(ss, moveText, '|') ||
            !std::getline(ss, scoreText, '|') ||
            !std::getline(ss, plyText, '|') ||
            !std::getline(ss, resultText))
        {
            std::cerr
                << "Malformed line "
                << lineNumber
                << "\n";

            ++rejected;
            continue;
        }

        fen        = trim(fen);
        moveText   = trim(moveText);
        scoreText  = trim(scoreText);
        plyText    = trim(plyText);
        resultText = trim(resultText);

        try
        {
            auto posOptional =
                chess::Position::tryFromFen(fen);

            if (!posOptional)
            {
                std::cerr
                    << "Invalid FEN line "
                    << lineNumber
                    << "\n";

                ++rejected;
                continue;
            }

            chess::Position pos = *posOptional;

            chess::Move move =
                chess::uci::uciToMove(pos, moveText);

            binpack::TrainingDataEntry entry;

            entry.pos    = pos;
            entry.move   = move;
            entry.score  =
                static_cast<std::int16_t>(
                    std::stoi(scoreText)
                );

            entry.ply =
                static_cast<std::uint16_t>(
                    std::stoul(plyText)
                );

            entry.result =
                static_cast<std::int16_t>(
                    std::stoi(resultText)
                );

            if (!entry.isValid())
            {
                std::cerr
                    << "Illegal move on line "
                    << lineNumber
                    << ": "
                    << moveText
                    << "\n";

                ++rejected;
                continue;
            }

            writer.addTrainingDataEntry(entry);
            ++written;
        }
        catch (const std::exception& e)
        {
            std::cerr
                << "Line "
                << lineNumber
                << ": "
                << e.what()
                << "\n";

            ++rejected;
        }
    }

    std::cout
        << "Written:  "
        << written
        << "\nRejected: "
        << rejected
        << "\n";

    return rejected == 0 ? 0 : 2;
}
'''

Path("/kaggle/working/nnue-pytorch/txt2binpack.cpp").write_text(code)

print("Created txt2binpack.cpp")

In [ ]:
%cd /kaggle/working/nnue-pytorch

!g++ -std=c++20 -O3 \
    -I. \
    txt2binpack.cpp \
    -o txt2binpack

In [ ]:
!rm -f /kaggle/working/test.binpack

!/kaggle/working/nnue-pytorch/txt2binpack \
    /kaggle/working/test_training.txt \
    /kaggle/working/test.binpack

!ls -lh /kaggle/working/test.binpack
!xxd -l 64 /kaggle/working/test.binpack

In [ ]:
SF = "/kaggle/working/stockfish/stockfish-linux-x86-64-universal"

engine = chess.engine.SimpleEngine.popen_uci(SF)

board = stockfish_randomized_opening(
    engine,
    min_plies=6,
    max_plies=12,
    nodes=2000,
    multipv=5
)

print(board)
print()
print("FEN:")
print(board.fen())

engine.quit()

In [ ]:
import os
from IPython.display import FileLink

# Zip your modified Stockfish source directory directly
os.system("zip -q -r /kaggle/working/Mac_tuned_sf19.zip /kaggle/working/stockfish")

# Print the download link
FileLink(r'Mac_tuned_sf19.zip')
'''
cd ~/Downloads
unzip -q Mac_tuned_sf19.zip
cd ~/Downloads/stockfish/src

# Clean old artifacts and compile natively for Mac Silicon
make clean
make profile-build ARCH=apple-silicon COMP=clang EvalFile=crackfish1924.nnue -j$(sysctl -n hw.ncpu)

# Apply Apple's free local memory signature to whitelist the binary
codesign --force --deep -s - ./stockfish
mv stockfish ~/Downloads/Crackfish1924_Mac


xattr -d com.apple.quarantine ~/Downloads/Mac_tuned_sf19 && chmod +x ~/Downloads/Mac_tuned_sf19

Run this on mac
'''

In [ ]:
!rm -rf /kaggle/working/Mac_tuned_sf19.zip

# 